# 第2章　科学計算とデータ処理（NumPy / pandas / matplotlib）

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 2.1　NumPy ― 画像は数字の配列である

In [ ]:
import numpy as np

# 512×512のゼロ埋め画像を作る
img = np.zeros((512, 512))
print(img.shape)          # → (512, 512)

# 配列の一部を切り出す（クロップ）
# Pythonのスライス a:b は b を含まない（半開区間）。だから長さは b−a になる。
patch = img[100:200, 100:200]   # 100行目から199行目まで＝100行分。終端の200は含まない

# 統計量を求める
print(img.mean(), img.std())    # 平均と標準偏差

# 値を正規化する（0〜1の範囲に収める）
img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)

## 2.2　3次元ボリュームをNumPyで扱う

In [ ]:
volume = np.load("ct_volume.npy")   # 例: 形は (150, 512, 512) = 150スライス
print(volume.shape)                 # → (150, 512, 512)

# 軸ごとに1枚の断面を取り出す（放射線の3方向に対応）
axial    = volume[75, :, :]     # 体の断面（アキシャル）：添字75のスライス
coronal  = volume[:, 256, :]    # 前後から見た冠状断（コロナル）
sagittal = volume[:, :, 256]    # 横から見た矢状断（サジタル）

# ボリューム全体の統計・正規化は2次元と同じ書き方でよい
print(volume.mean(), volume.min(), volume.max())

## 手を動かす ― 小さな「ミニCT」でクロップとウィンドウを確かめる

In [ ]:
import numpy as np

# 6×6の「ミニCT」。値はHUに見立てる（-1000=空気, 40前後=軟部, 950前後=骨）
mini = np.array([
    [-1000, -1000, -900, -900, -1000, -1000],
    [-1000,    30,   40,   45,    30, -1000],
    [ -900,    40,   50,  900,    45,  -900],
    [ -900,    45,  950,  960,    40,  -900],
    [-1000,    35,   45,   40,    35, -1000],
    [-1000, -1000, -900, -900, -1000, -1000],
])
print(mini.shape)                 # (6, 6)

# ① クロップ＝ただのスライス。中央の4×4を切り出す
center = mini[1:5, 1:5]
print(center.shape)               # (4, 4)

# ② 軟部組織ウィンドウ（center=40, width=400 → -160〜240 を 0〜1 に）
low, high = 40 - 200, 40 + 200
windowed = np.clip(mini, low, high)          # 範囲外を切り詰める
windowed = (windowed - low) / (high - low)   # 0〜1に正規化
print(round(windowed.min(), 2), round(windowed.max(), 2))   # 0.0 1.0
print(np.round(windowed[3], 2))  # → [0.   0.51 1.   1.   0.5  0.  ]

## 数字で追う ― 軸を指定した集計と keepdims で「スライスごとに正規化」する

In [ ]:
import numpy as np

# (スライス, 高さ, 幅) = (3, 2, 2)。スライスごとに明るさが大きく違う
vol = np.array([
    [[ 10,  20], [ 30,  40]],   # slice0：暗い
    [[100, 120], [140, 160]],   # slice1：明るい
    [[  0,   4], [  8,  12]],   # slice2：さらに暗い
], dtype=np.float32)

# ① ボリューム全体で一つの平均（スカラー）
print(vol.mean())                       # 53.666668

# ② スライスごとの平均は「高さ・幅の軸」だけを畳む → axis=(1,2)
print(vol.mean(axis=(1, 2)))            # [ 25. 130.   6.]  形は (3,)
print(vol.mean(axis=(1, 2), keepdims=True).shape)   # (3, 1, 1)

In [ ]:
mean = vol.mean(axis=(1, 2), keepdims=True)   # (3, 1, 1)
std  = vol.std(axis=(1, 2), keepdims=True)    # (3, 1, 1)
z = (vol - mean) / std                         # スライスごとにz-score正規化
print(np.round(z[0], 2))   # [[-1.34 -0.45] [ 0.45  1.34]]
print(np.round(z.mean(axis=(1, 2)), 2))        # [ 0.  0.  0.]（各スライス平均0）

## 2.3　医療画像特有の前処理

In [ ]:
def apply_ct_window(img_hu, center, width):
    """CT画像にウィンドウ処理を適用する"""
    low = center - width / 2
    high = center + width / 2
    img = np.clip(img_hu, low, high)     # 範囲外を切り詰める
    return (img - low) / (high - low)    # 0〜1に正規化

# 腹部軟部組織用のウィンドウ
img_window = apply_ct_window(ct_hu, center=40, width=400)

## 2.4　pandas ― 症例データを表で管理する

In [ ]:
import pandas as pd

# CSVを読み込む
df = pd.read_csv("dataset_mapping.csv")
print(df.head())               # 先頭5行を表示
print(df["Disease_Label"].value_counts())  # ラベルごとの症例数

# 条件でしぼり込む
train_df = df[df["split"] == "train"]

# 新しいCSVとして保存
train_df.to_csv("train.csv", index=False)

## 手を動かす ― 症例単位で分割する（リークを防ぐ最重要作法）

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 同じ患者が複数枚の画像を持つ（両眼・複数スライスなど）
df = pd.DataFrame({
    "image_id":   ["a1", "a2", "b1", "c1", "c2", "d1"],
    "patient_id": ["P01", "P01", "P02", "P03", "P03", "P04"],
    "label":      [1, 1, 0, 1, 1, 0],
})

# 患者ID（groups）ごと、まるごと学習/検証に振り分ける
splitter = GroupShuffleSplit(n_splits=1, test_size=0.34, random_state=0)
train_idx, val_idx = next(splitter.split(df, groups=df["patient_id"]))
train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]

# 患者IDが両者で重ならないことを、機械的に必ず確認する
overlap = set(train_df["patient_id"]) & set(val_df["patient_id"])
print("重複患者:", overlap)          # → set()（空集合＝リークなし）

## 2.5　matplotlib ― 画像と結果を可視化する

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("input")
axes[1].imshow(mask, cmap="jet", alpha=0.5)
axes[1].set_title("ground truth")
axes[2].imshow(image, cmap="gray")
axes[2].imshow(prediction, cmap="jet", alpha=0.5)  # 元画像に重ねる
axes[2].set_title("prediction (overlay)")
plt.savefig("result.png", dpi=150)